# 04 — Análise de Correspondência Múltipla (Cap. 6.4)Mapeia a circulação de motivos iconográficos entre países, períodos e regimes.**Método:** MCA via `prince`

In [ ]:
import pandas as pd, numpy as np, princeimport matplotlib.pyplot as plt, seaborn as snssns.set_theme(style="whitegrid", font_scale=1.1)df = pd.read_csv('../data/processed/corpus_dataset.csv')df['endurec_level'] = pd.qcut(df['purificacao_composto'], q=3, labels=['baixo','medio','alto'])mca_cols = ['country', 'period_norm', 'medium_norm', 'regime_iconocratico', 'endurec_level']df_mca = df[mca_cols].copy()country_counts = df_mca['country'].value_counts()df_mca.loc[df_mca['country'].isin(country_counts[country_counts < 4].index), 'country'] = 'Other'df_mca = df_mca.dropna()print(f"MCA input: {len(df_mca)} itens, {len(mca_cols)} variáveis")for col in mca_cols: print(f"  {col}: {df_mca[col].nunique()} categorias")

## 4.1 MCA — Inércia explicada

In [ ]:
mca = prince.MCA(n_components=3, random_state=42)mca = mca.fit(df_mca)print("Inércia explicada por componente:")for i, (eig, pct) in enumerate(zip(mca.eigenvalues_, mca.percentage_of_variance_)):    print(f"  Dim {i+1}: eigenvalue = {eig:.4f}, inércia = {pct:.1f}%")print(f"  Total (3 dims): {sum(mca.percentage_of_variance_[:3]):.1f}%")

## 4.2 Biplot: categorias no espaço MCA

In [ ]:
col_coords = mca.column_coordinates(df_mca)var_colors = {'country': '#e74c3c', 'period_norm': '#2ecc71', 'medium_norm': '#9b59b6', 'regime_iconocratico': '#3498db', 'endurec_level': '#f39c12'}fig, ax = plt.subplots(figsize=(12, 10))for var in mca_cols:    mask = col_coords.index.str.startswith(var + '_')    coords = col_coords[mask]    ax.scatter(coords[0], coords[1], s=100, color=var_colors[var], label=var, zorder=5, alpha=0.8)    for idx, row in coords.iterrows():        label = idx.replace(var + '_', '')        ax.annotate(label, (row[0], row[1]), fontsize=8, ha='center', va='bottom', color=var_colors[var], fontweight='bold')ax.axhline(0, color='grey', ls='--', alpha=0.3); ax.axvline(0, color='grey', ls='--', alpha=0.3)ax.set_xlabel(f"Dim 1 ({mca.percentage_of_variance_[0]:.1f}%)"); ax.set_ylabel(f"Dim 2 ({mca.percentage_of_variance_[1]:.1f}%)")ax.set_title('MCA Biplot — Categorias no espaço iconocrático'); ax.legend(loc='upper right', framealpha=0.9)plt.tight_layout(); plt.savefig('../data/processed/fig_09_mca_biplot.png', dpi=150, bbox_inches='tight'); plt.show()

## 4.3 Itens individuais no espaço MCA

In [ ]:
row_coords = mca.row_coordinates(df_mca)row_coords['regime'] = df_mca['regime_iconocratico'].valuesrow_coords['id'] = df.loc[df_mca.index, 'id'].valuesregime_palette = {'fundacional': '#e74c3c', 'normativo': '#3498db', 'militar': '#2c3e50'}fig, ax = plt.subplots(figsize=(12, 10))for regime, color in regime_palette.items():    mask = row_coords['regime'] == regime; sub = row_coords[mask]    ax.scatter(sub[0], sub[1], c=color, label=regime.capitalize(), alpha=0.6, s=40, edgecolors='white', linewidth=0.5)threshold = row_coords[[0, 1]].abs().max(axis=1).quantile(0.9)outliers = row_coords[row_coords[[0, 1]].abs().max(axis=1) > threshold]for _, row in outliers.iterrows(): ax.annotate(row['id'], (row[0], row[1]), fontsize=7, alpha=0.7)ax.axhline(0, color='grey', ls='--', alpha=0.3); ax.axvline(0, color='grey', ls='--', alpha=0.3)ax.set_xlabel(f"Dim 1 ({mca.percentage_of_variance_[0]:.1f}%)"); ax.set_ylabel(f"Dim 2 ({mca.percentage_of_variance_[1]:.1f}%)")ax.set_title('Itens do corpus no espaço MCA (coloridos por regime)'); ax.legend()plt.tight_layout(); plt.savefig('../data/processed/fig_10_mca_individuals.png', dpi=150, bbox_inches='tight'); plt.show()row_coords['dist'] = np.sqrt(row_coords[0]**2 + row_coords[1]**2)print("\nItens extremos (top 10):")for _, r in row_coords.nlargest(10, 'dist').iterrows(): print(f"  {r['id']:20s} regime={r['regime']:12s} dist={r['dist']:.2f}")